In [ ]:
import json
from pathlib import Path
from datetime import date

import pandas as pd
import yaml
import numpy as np

from common import read_day, daterange

with open("../config.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [9]:
days = (
    list(daterange(date(2023, 10, 1), date(2023, 12, 31)))
    + list(daterange(date(2024, 1, 1), date(2024, 2, 29)))
    + list(daterange(date(2024, 10, 1), date(2024, 12, 31)))
    + list(daterange(date(2025, 1, 1), date(2025, 12, 31)))
)

In [ ]:
ids = []
for i in days:
    temp = read_day(i, config["dirs"]["raw_source"])
    dids = temp.unique(subset=["device_aid_hashed"], keep="first")[
        "device_aid_hashed"
    ].to_list()
    ids += dids
# runs 25 minutes

In [11]:
len(ids)

33243659

In [12]:
len(np.unique(ids))

1652460

In [13]:
lookup = {}
for i, device_id in enumerate(np.unique(ids)):
    lookup[device_id] = i + 1

In [14]:
Path("../output/id_lookup").mkdir(parents=True, exist_ok=True)
with open("../output/id_lookup/id_lookup.json", "w") as fp:
    json.dump(lookup, fp, indent=1)

## create partitions from original IDs

In [15]:
with open("../output/id_lookup/id_lookup.json", "r") as fp:
    lookup = json.load(fp)

In [16]:
partition = {v: k[:2] for k, v in lookup.items()}

In [17]:
with open("../output/id_lookup/partition.json", "w") as fp:
    json.dump(partition, fp, indent=1)

In [18]:
partition_distribution = (
    pd.DataFrame({"partition": partition.values(), "device_id": partition.keys()})
    .groupby("partition")["device_id"]
    .count()
    .reset_index()
)

In [19]:
partition_distribution["device_id"].describe()

count     256.000000
mean     6454.921875
std       114.638262
min      6153.000000
25%      6382.000000
50%      6457.500000
75%      6536.250000
max      6765.000000
Name: device_id, dtype: float64